# `apply` vs `apply_peetre`: Efficiency & Precision Analysis (Targeted Backends)

This notebook benchmarks the Peetre decomposition pipeline with a **targeted
backend strategy**: each joint-residual symbol is only paired with the backends
that are mathematically suited to its structure.

### Why targeted?

Running every backend on every symbol wastes time and is logically inconsistent:

| Symbol class | Natural backend | Why others fail |
|---|---|---|
| Smooth Gaussian `exp(−(xξ)²/w)` | `lowrank` | NUFFT has no oscillatory phase; AAA has no poles |
| Rational `1/(1+x²+ξ²)` | `aaa` | NUFFT has no phase; lowrank converges poorly on peaks |
| Oscillatory `sin(xξ)` | `nufft` | AAA cannot fit rapid oscillations; lowrank needs many terms |
| Mixed (all three) | `hybrid` | Monolithic `auto` sees the sum, finds no global pattern → falls back to `direct` |

By running only the appropriate backends, we:
1. **Eliminate wasted fallback time** (e.g. NUFFT attempting and failing on a rational symbol).
2. **Produce meaningful error comparisons** (each backend is tested on its target class).
3. **Reduce total wall-clock time** by roughly 50–70% compared to the all-backends approach.

**Boundary condition**: `periodic` throughout (NUFFT requires it; all other
backends work with both periodic and dirichlet).


### 🐛 Bug Fix Note: The Frequency Grid Misalignment

If you previously ran this with `np.fft.fftshift` applied to `kx` and `ky`,
you likely saw massive errors (e.g., `1.46e+04`) for Variable Coefficients,
while Constant Coefficients had `0.00e+00` error.

**Why?** The slow path in `apply()` correctly recomputes frequencies internally.
But the fast path `_apply_constant_fft()` (used by `apply_peetre()` for the
separated $q(\xi)$ multipliers) blindly uses the `kx` array passed to it.
Passing a **shifted** `kx` multiplied the unshifted FFT spectrum by the shifted
symbol, completely misaligning the frequencies.

**The Fix:** We simply use the **unshifted** `fftfreq` grid below so it perfectly
matches the `scipy.fft.fft` convention.


In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import sympy as sp
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
from psiop import PseudoDifferentialOperator

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

boundary_condition = 'periodic'  # NUFFT requires periodic


## Benchmark harness with targeted backend selection

The key improvement over the all-backends approach is `get_configs_for_symbol()`,
which returns only the backends that are mathematically appropriate for each
symbol's structural class.

| Symbol class | Backends tested |
|---|---|
| Local / Non-Local (no joint residual) | `direct`, `peetre` |
| Joint (Smooth Gaussian) | `direct`, `peetre(lowrank)`, `peetre(auto)` |
| Joint (Resolvent / Mixed local+resolvent) | `direct`, `peetre(aaa)`, `peetre(auto)` |
| Joint (Oscillatory / Chirp) | `direct`, `peetre(nufft)`, `peetre(auto)` |
| Joint (AllThree mixed) | `direct`, `peetre(auto)`, `hybrid` |

This avoids running NUFFT on rational symbols, AAA on oscillatory symbols,
etc., which would just trigger expensive fallbacks to direct quadrature.


In [ ]:
def make_test_function(op, x_grid, y_grid=None):
    """Same test function u as in the baseline efficiency analysis."""
    if op.dim == 1:
        return np.exp(-x_grid**2) * np.cos(5 * x_grid)
    X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')
    return np.exp(-(X**2 + Y**2)) * np.cos(5 * X) * np.cos(5 * Y)


def get_configs_for_symbol(name):
    """
    Return only the backends that are mathematically appropriate
    for this symbol's structural class.

    This avoids running e.g. NUFFT on a rational symbol (which would
    fail and fall back to direct, wasting time), or AAA on an
    oscillatory symbol (which would produce a poor fit and fall back).
    """
    if 'Local' in name or 'Non-Local' in name:
        # No joint residual: direct vs peetre (exact, O(N log N))
        return [
            {'label': 'direct',
             'kwargs': dict(backend='direct')},
            {'label': 'peetre (no joint)',
             'kwargs': dict(backend='peetre')},
        ]
    elif 'Smooth' in name:
        # Smooth joint kernel: lowrank is the natural fit
        return [
            {'label': 'direct',
             'kwargs': dict(backend='direct')},
            {'label': 'peetre (joint=lowrank)',
             'kwargs': dict(backend='peetre', joint_backend='lowrank')},
            {'label': 'peetre (joint=auto)',
             'kwargs': dict(backend='peetre', joint_backend='auto')},
        ]
    elif 'Resolvent' in name or 'Mixed' in name:
        # Rational / pole structure: aaa is the natural fit
        return [
            {'label': 'direct',
             'kwargs': dict(backend='direct')},
            {'label': 'peetre (joint=aaa)',
             'kwargs': dict(backend='peetre', joint_backend='aaa')},
            {'label': 'peetre (joint=auto)',
             'kwargs': dict(backend='peetre', joint_backend='auto')},
        ]
    elif 'Oscillatory' in name or 'Chirp' in name:
        # Oscillatory phase: nufft is the natural fit
        return [
            {'label': 'direct',
             'kwargs': dict(backend='direct')},
            {'label': 'peetre (joint=nufft)',
             'kwargs': dict(backend='peetre', joint_backend='nufft')},
            {'label': 'peetre (joint=auto)',
             'kwargs': dict(backend='peetre', joint_backend='auto')},
        ]
    elif 'AllThree' in name:
        # Mixed structure: monolithic auto falls back to direct;
        # hybrid per-term routing is the correct approach.
        # hybrid is handled separately via apply_hybrid() in the loop.
        return [
            {'label': 'direct',
             'kwargs': dict(backend='direct')},
            {'label': 'peetre (joint=auto)',
             'kwargs': dict(backend='peetre', joint_backend='auto')},
        ]
    else:
        # Fallback: run direct + auto
        return [
            {'label': 'direct',
             'kwargs': dict(backend='direct')},
            {'label': 'peetre (joint=auto)',
             'kwargs': dict(backend='peetre', joint_backend='auto')},
        ]


def benchmark_configs(op, u, x_grid, kx, y_grid=None, ky=None, configs=None,
                      repeats=3, boundary_condition='periodic'):
    """
    Run each configuration in `configs` on `op.apply(u, ...)`, timing it and
    computing its relative L2 error against the first configuration's output
    (assumed to be backend='direct', i.e. ground truth).

    Returns
    -------
    dict: label -> {'mean_time': float, 'times': list[float], 'rel_l2_error': float}
    """
    if configs is None:
        configs = [{'label': 'direct', 'kwargs': dict(backend='direct')}]

    results = {}
    ref = None

    for cfg in configs:
        label, kwargs = cfg['label'], cfg['kwargs']

        # Warm-up call
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            op.apply(u, x_grid, kx, y_grid=y_grid, ky=ky,
                     boundary_condition=boundary_condition, **kwargs)

        times = []
        out = None
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            for _ in range(repeats):
                t0 = time.perf_counter()
                out = op.apply(u, x_grid, kx, y_grid=y_grid, ky=ky,
                               boundary_condition=boundary_condition, **kwargs)
                times.append(time.perf_counter() - t0)

        mean_t = float(np.mean(times))

        if ref is None:
            ref = out
            err = 0.0
        else:
            norm_ref = np.linalg.norm(ref)
            err = float(np.linalg.norm(out - ref) / norm_ref) if norm_ref > 0 else float(np.linalg.norm(out - ref))

        results[label] = {'mean_time': mean_t, 'times': times, 'rel_l2_error': err}

    return results


def benchmark_hybrid(op, u, x_grid, kx, y_grid=None, ky=None,
                     repeats=3, boundary_condition='periodic'):
    """
    Compare monolithic auto vs apply_hybrid for mixed-structure symbols.

    Returns
    -------
    dict with keys 'monolithic' and 'hybrid', each containing
    {'mean_time', 'rel_l2_error'} relative to direct ground truth.
    """
    # Ground truth
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        ref = op.apply(u, x_grid, kx, y_grid=y_grid, ky=ky,
                       boundary_condition=boundary_condition,
                       backend='direct')

    results = {}
    for label, fn in [
        ('monolithic (auto)', lambda: op.apply(
            u, x_grid, kx, y_grid=y_grid, ky=ky,
            boundary_condition=boundary_condition,
            backend='peetre', joint_backend='auto')),
        ('hybrid', lambda: op.apply_hybrid(
            u, x_grid, kx, y_grid=y_grid, ky=ky,
            boundary_condition=boundary_condition)),
    ]:
        # Warm-up
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            fn()

        times = []
        out = None
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            for _ in range(repeats):
                t0 = time.perf_counter()
                out = fn()
                times.append(time.perf_counter() - t0)

        norm_ref = np.linalg.norm(ref)
        err = float(np.linalg.norm(out - ref) / norm_ref) if norm_ref > 0 else 0.0
        results[label] = {
            'mean_time': float(np.mean(times)),
            'rel_l2_error': err,
        }
    return results


## 1D Symbols

Each symbol is annotated with its structural class and the backend(s)
that will be tested against it:

| Symbol | Class | Backends tested |
|---|---|---|
| `ξ²` | Local (Const) | direct, peetre |
| `(1+0.5 sin x)·ξ²` | Local (Var) | direct, peetre |
| `√(ξ²+1)` | Non-Local (Const) | direct, peetre |
| `(1+0.5 sin x)·√(ξ²+1)` | Non-Local (Var) | direct, peetre |
| `exp(−(xξ)²/40000)` | Joint (Smooth) | direct, **lowrank**, auto |
| `1/(1+x²+ξ²)` | Joint (Resolvent) | direct, **aaa**, auto |
| `ξ² + 1/(1+x²+ξ²)` | Joint (Mixed) | direct, **aaa**, auto |
| `sin(x·ξ)` | Joint (Oscillatory) | direct, **nufft**, auto |
| `x²·exp(ixξ)·cos(ξ)` | Joint (Chirp) | direct, **nufft**, auto |
| `sin(xξ)+1/(1+(x−ξ)²)+exp(−(x−ξ)²/8)` | Joint (AllThree) | direct, auto, **hybrid** |


In [ ]:
# 1D Symbols
x, xi = sp.symbols('x xi', real=True)

sym_1d_loc_c   = xi**2
sym_1d_loc_v   = (1 + 0.5 * sp.sin(x)) * xi**2
sym_1d_nloc_c  = sp.sqrt(xi**2 + 1.0)
sym_1d_nloc_v  = (1 + 0.5 * sp.sin(x)) * sp.sqrt(xi**2 + 1.0)

# Joint symbols: entangled in x and xi, cannot be separated into a(x) * q(xi)
sym_1d_joint_smooth    = sp.exp(-(x * xi)**2 / 40000)
sym_1d_joint_resolvent = 1 / (1 + x**2 + xi**2)
sym_1d_joint_mixed     = xi**2 + 1 / (1 + x**2 + xi**2)

# Oscillatory symbols for NUFFT
sym_1d_joint_osc       = sp.sin(x * xi)
sym_1d_joint_chirp     = x**2 * sp.exp(sp.I * x * xi) * sp.cos(xi)

# Mixed-structure symbol (all three backend classes)
sym_1d_joint_allthree  = (sp.sin(x * xi)
                          + 1 / (1 + (x - xi)**2)
                          + sp.exp(-((x - xi)**2) / 8))

ops_1d = {
    '1D Local (Const)':       PseudoDifferentialOperator(sym_1d_loc_c, [x], mode='symbol'),
    '1D Local (Var)':         PseudoDifferentialOperator(sym_1d_loc_v, [x], mode='symbol'),
    '1D Non-Local (Const)':   PseudoDifferentialOperator(sym_1d_nloc_c, [x], mode='symbol'),
    '1D Non-Local (Var)':     PseudoDifferentialOperator(sym_1d_nloc_v, [x], mode='symbol'),
    '1D Joint (Smooth)':      PseudoDifferentialOperator(sym_1d_joint_smooth, [x], mode='symbol'),
    '1D Joint (Resolvent)':   PseudoDifferentialOperator(sym_1d_joint_resolvent, [x], mode='symbol'),
    '1D Joint (Mixed)':       PseudoDifferentialOperator(sym_1d_joint_mixed, [x], mode='symbol'),
    '1D Joint (Oscillatory)': PseudoDifferentialOperator(sym_1d_joint_osc, [x], mode='symbol'),
    '1D Joint (Chirp)':       PseudoDifferentialOperator(sym_1d_joint_chirp, [x], mode='symbol'),
    '1D Joint (AllThree)':    PseudoDifferentialOperator(sym_1d_joint_allthree, [x], mode='symbol'),
}


In [ ]:
N_values_1d = [16, 32, 64, 128, 256, 512]

rows_1d = []

print('Running 1D Benchmarks (targeted backends)...')
for N in N_values_1d:
    L = 10
    dx = L / N
    x_grid = -L/2 + dx * np.arange(N)

    # Unshifted frequencies to match scipy.fft.fft output convention
    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)

    for name, op in ops_1d.items():
        u = make_test_function(op, x_grid)
        cfgs = get_configs_for_symbol(name)
        res = benchmark_configs(op, u, x_grid, kx, repeats=4,
                                boundary_condition=boundary_condition,
                                configs=cfgs)

        t_direct = res['direct']['mean_time']

        for label, r in res.items():
            speedup = t_direct / r['mean_time'] if r['mean_time'] > 0 else np.nan
            rows_1d.append(dict(
                N=N, symbol=name, backend=label,
                time=r['mean_time'], speedup=speedup, err=r['rel_l2_error']
            ))
            print(f"N={N:5d} | {name:24s} | {label:24s} | "
                  f"time: {r['mean_time']:.4f}s | err: {r['rel_l2_error']:.2e}")

        # Special handling for AllThree: also run hybrid
        if 'AllThree' in name:
            hyb_res = benchmark_hybrid(op, u, x_grid, kx,
                                       repeats=4,
                                       boundary_condition=boundary_condition)
            for label, r in hyb_res.items():
                speedup = t_direct / r['mean_time'] if r['mean_time'] > 0 else np.nan
                rows_1d.append(dict(
                    N=N, symbol=name, backend=label,
                    time=r['mean_time'], speedup=speedup, err=r['rel_l2_error']
                ))
                print(f"N={N:5d} | {name:24s} | {label:24s} | "
                      f"time: {r['mean_time']:.4f}s | err: {r['rel_l2_error']:.2e}")

df_1d = pd.DataFrame(rows_1d)


### 1D Results Table


In [ ]:
pivot_time_1d = df_1d.pivot_table(index=['symbol', 'N'], columns='backend', values='time')
pivot_err_1d  = df_1d.pivot_table(index=['symbol', 'N'], columns='backend', values='err')
pivot_speedup_1d = df_1d.pivot_table(index=['symbol', 'N'], columns='backend', values='speedup')

display(pivot_time_1d)
display(pivot_err_1d)
display(pivot_speedup_1d)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

names_1d = list(ops_1d.keys())
cmap = plt.get_cmap('tab10')
colors_1d = {name: cmap(i % 10) for i, name in enumerate(names_1d)}

# Collect all unique backend labels across all symbols
all_labels_1d = sorted(df_1d['backend'].unique())
linestyles = {l: ['--', '-', ':', '-.', (0, (3, 1, 1, 1)), (0, (5, 2))][i % 6]
              for i, l in enumerate(all_labels_1d)}
markers    = {l: ['o', 's', '^', 'D', 'v', 'P'][i % 6]
              for i, l in enumerate(all_labels_1d)}

for name in names_1d:
    sub = df_1d[df_1d['symbol'] == name]
    for label in all_labels_1d:
        s = sub[sub['backend'] == label].sort_values('N')
        if len(s) == 0:
            continue
        ax1.loglog(s['N'], s['time'], marker=markers[label],
                   linestyle=linestyles[label],
                   color=colors_1d[name], label=f'{name} ({label})', alpha=0.85)

ax1.set_xlabel('Grid Size N')
ax1.set_ylabel('Execution Time (s)')
ax1.set_title('1D Execution Time: targeted backends')
ax1.legend(fontsize=5, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))

for name in names_1d:
    sub = df_1d[df_1d['symbol'] == name]
    for label in all_labels_1d:
        if label == 'direct':
            continue  # skip direct (err=0)
        s = sub[sub['backend'] == label].sort_values('N')
        if len(s) == 0:
            continue
        ax2.semilogy(s['N'], s['err'].clip(lower=1e-17), marker=markers[label],
                     linestyle=linestyles[label], color=colors_1d[name],
                     label=f'{name} ({label})', alpha=0.85)

ax2.set_xlabel('Grid Size N')
ax2.set_ylabel('Relative L2 Error (vs. direct)')
ax2.set_title('1D Precision: targeted backends vs direct ground truth')
ax2.legend(fontsize=5, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))

plt.tight_layout()
plt.show()


---
## Backend Classification via `classify_joint=True`

`peetre_decomposition(classify_joint=True)` runs the symbolic auto-selector
at decomposition time and stores the recommended backend in
`deco['joint_backend']`. This is purely symbolic — no grid, no bounds.

The table below verifies that the auto-selector recommends the correct
backend for each 1D joint symbol, confirming our targeted assignment.


In [ ]:
print(f"{'Symbol':<28s} {'Recommended backend':<20s}")
print('-' * 50)
for name, op in ops_1d.items():
    deco = op.peetre_decomposition(classify_joint=True)
    jb = deco.get('joint_backend', '— (no joint residual)')
    print(f"{name:<28s} {jb:<20s}")


---
## Joint Backend Deep-Dive (1D)

For each structural class of joint symbol, we compare the **target** backend
against direct at a fixed grid size. This isolates the approximation quality
and speed of each backend on its intended class.

| Symbol class | Target backend | Also tested |
|---|---|---|
| Oscillatory (`sin(xξ)`) | `nufft` | auto |
| Resolvent (`1/(1+x²+ξ²)`) | `aaa` | auto |
| Smooth Gaussian (`exp(−(xξ)²/40000)`) | `lowrank` | auto |
| Chirp (`x²·exp(ixξ)·cos(ξ)`) | `nufft` | auto |


In [ ]:
N_dd = 512
L_dd = 10
dx_dd = L_dd / N_dd
x_dd = -L_dd/2 + dx_dd * np.arange(N_dd)
kx_dd = 2 * np.pi * np.fft.fftfreq(N_dd, d=dx_dd)
u_dd = make_test_function(ops_1d['1D Joint (Smooth)'], x_dd)

# Only test the target backend + auto for each class
deepdive_symbols = {
    'Oscillatory':  (sym_1d_joint_osc,       ['nufft', 'auto']),
    'Resolvent':    (sym_1d_joint_resolvent,  ['aaa', 'auto']),
    'Smooth':       (sym_1d_joint_smooth,     ['lowrank', 'auto']),
    'Chirp':        (sym_1d_joint_chirp,      ['nufft', 'auto']),
}

print(f"{'Class':<14s} {'Backend':<12s} {'Time (s)':>10s} {'Rel L2 err':>12s}")
print('=' * 52)

dd_rows = []
for cls_name, (sym_expr, backends) in deepdive_symbols.items():
    op_dd = PseudoDifferentialOperator(sym_expr, [x], mode='symbol')
    cfgs = [{'label': 'direct', 'kwargs': dict(backend='direct')}]
    for jb in backends:
        cfgs.append({'label': jb, 'kwargs': dict(backend='peetre', joint_backend=jb)})

    res = benchmark_configs(op_dd, u_dd, x_dd, kx_dd, configs=cfgs,
                            repeats=3, boundary_condition=boundary_condition)
    for label, r in res.items():
        print(f"{cls_name:<14s} {label:<12s} {r['mean_time']:>10.4f} {r['rel_l2_error']:>12.3e}")
        dd_rows.append(dict(cls=cls_name, backend=label,
                            time=r['mean_time'], err=r['rel_l2_error']))
    print()

df_dd = pd.DataFrame(dd_rows)


---
## Hybrid vs Monolithic Auto-Routing (1D)

For the mixed-structure symbol
`sin(xξ) + 1/(1+(x−ξ)²) + exp(−(x−ξ)²/8)`,
monolithic `joint_backend='auto'` sees the **sum** and cannot find a single
global pattern — it falls back to `direct` O(N²) quadrature.

`apply_hybrid()` splits the joint residual into additive terms and routes
each one to its optimal backend (NUFFT, AAA, lowrank), preserving the
O(N log N) speedup.


In [ ]:
N_hyb = 512
L_hyb = 10
dx_hyb = L_hyb / N_hyb
x_hyb = -L_hyb/2 + dx_hyb * np.arange(N_hyb)
kx_hyb = 2 * np.pi * np.fft.fftfreq(N_hyb, d=dx_hyb)
u_hyb = make_test_function(ops_1d['1D Joint (AllThree)'], x_hyb)

op_hyb = ops_1d['1D Joint (AllThree)']

print(f"{'Backend':<22s} {'Time (s)':>10s} {'Rel L2 err':>12s}")
print('=' * 48)

hyb_res = benchmark_hybrid(op_hyb, u_hyb, x_hyb, kx_hyb,
                           repeats=3, boundary_condition=boundary_condition)
for label, r in hyb_res.items():
    print(f"{label:<22s} {r['mean_time']:>10.4f} {r['rel_l2_error']:>12.3e}")

# Also show direct for reference
t0 = time.perf_counter()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _ = op_hyb.apply(u_hyb, x_hyb, kx_hyb,
                     boundary_condition=boundary_condition, backend='direct')
t_dir = time.perf_counter() - t0
print(f"{'direct (reference)':<22s} {t_dir:>10.4f} {'0.000e+00':>12s}")


## 2D Symbols

Same structure as 1D, with genuinely joint 2D symbols added. We also add
an **oscillatory 2D** symbol to exercise the 2D NUFFT backend.

As in the original notebook, `N` is kept modest for 2D because
`apply(backend="direct")` on spatially-dependent symbols is O(N⁴).


In [ ]:
# 2D Symbols
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

sym_2d_loc_c  = xi**2 + eta**2
sym_2d_loc_v  = (1 + 0.5 * sp.sin(x) * sp.cos(y)) * (xi**2 + eta**2)
sym_2d_nloc_c = (xi**2 + eta**2)**0.75
sym_2d_nloc_v = (1 + 0.5 * sp.sin(x) * sp.cos(y)) * (xi**2 + eta**2)**0.75

# Joint symbols
sym_2d_joint_smooth    = sp.exp(-((x * xi)**2 + (y * eta)**2) / 40000)
sym_2d_joint_resolvent = 1 / (1 + x**2 + y**2 + xi**2 + eta**2)
sym_2d_joint_osc       = sp.sin((x + y) * xi)  # NUFFT-friendly 2D

ops_2d = {
    '2D Local (Const)':       PseudoDifferentialOperator(sym_2d_loc_c, [x, y], mode='symbol'),
    '2D Local (Var)':         PseudoDifferentialOperator(sym_2d_loc_v, [x, y], mode='symbol'),
    '2D Non-Local (Const)':   PseudoDifferentialOperator(sym_2d_nloc_c, [x, y], mode='symbol'),
    '2D Non-Local (Var)':     PseudoDifferentialOperator(sym_2d_nloc_v, [x, y], mode='symbol'),
    '2D Joint (Smooth)':      PseudoDifferentialOperator(sym_2d_joint_smooth, [x, y], mode='symbol'),
    '2D Joint (Resolvent)':   PseudoDifferentialOperator(sym_2d_joint_resolvent, [x, y], mode='symbol'),
    '2D Joint (Oscillatory)': PseudoDifferentialOperator(sym_2d_joint_osc, [x, y], mode='symbol'),
}


In [ ]:
# We limit N to 128 for 2D because `apply(backend='direct')` on variable / joint symbols
# is O(N^4) and would take too long / OOM for N=256+.
N_values_2d = [16, 32, 64, 128]

rows_2d = []

print('Running 2D Benchmarks (targeted backends)...')
for N in N_values_2d:
    L = 10
    dx = L / N
    x_grid = -L/2 + dx * np.arange(N)
    y_grid = -L/2 + dx * np.arange(N)

    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(N, d=dx)

    for name, op in ops_2d.items():
        u = make_test_function(op, x_grid, y_grid)
        cfgs = get_configs_for_symbol(name)
        res = benchmark_configs(op, u, x_grid, kx, y_grid=y_grid, ky=ky,
                                repeats=2, boundary_condition=boundary_condition,
                                configs=cfgs)

        t_direct = res['direct']['mean_time']

        for label, r in res.items():
            speedup = t_direct / r['mean_time'] if r['mean_time'] > 0 else np.nan
            rows_2d.append(dict(
                N=N, symbol=name, backend=label,
                time=r['mean_time'], speedup=speedup, err=r['rel_l2_error']
            ))
            print(f"N={N:4d}x{N:<4d} | {name:24s} | {label:24s} | "
                  f"time: {r['mean_time']:.4f}s | err: {r['rel_l2_error']:.2e}")

df_2d = pd.DataFrame(rows_2d)


### 2D Results Table


In [ ]:
pivot_time_2d = df_2d.pivot_table(index=['symbol', 'N'], columns='backend', values='time')
pivot_err_2d  = df_2d.pivot_table(index=['symbol', 'N'], columns='backend', values='err')
pivot_speedup_2d = df_2d.pivot_table(index=['symbol', 'N'], columns='backend', values='speedup')

display(pivot_time_2d)
display(pivot_err_2d)
display(pivot_speedup_2d)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

names_2d = list(ops_2d.keys())
colors_2d = {name: cmap(i % 10) for i, name in enumerate(names_2d)}

all_labels_2d = sorted(df_2d['backend'].unique())

for name in names_2d:
    sub = df_2d[df_2d['symbol'] == name]
    for label in all_labels_2d:
        s = sub[sub['backend'] == label].sort_values('N')
        if len(s) == 0:
            continue
        ax1.loglog(s['N'], s['time'], marker=markers.get(label, 'o'),
                   linestyle=linestyles.get(label, '-'),
                   color=colors_2d[name], label=f'{name} ({label})', alpha=0.85)

ax1.set_xlabel('Grid Size N (Total Points = N²)')
ax1.set_ylabel('Execution Time (s)')
ax1.set_title('2D Execution Time: targeted backends')
ax1.legend(fontsize=5, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))

for name in names_2d:
    sub = df_2d[df_2d['symbol'] == name]
    for label in all_labels_2d:
        if label == 'direct':
            continue
        s = sub[sub['backend'] == label].sort_values('N')
        if len(s) == 0:
            continue
        ax2.semilogy(s['N'], s['err'].clip(lower=1e-17), marker=markers.get(label, 'o'),
                     linestyle=linestyles.get(label, '-'), color=colors_2d[name],
                     label=f'{name} ({label})', alpha=0.85)

ax2.set_xlabel('Grid Size N')
ax2.set_ylabel('Relative L2 Error (vs. direct)')
ax2.set_title('2D Precision: targeted backends vs direct ground truth')
ax2.legend(fontsize=5, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))

plt.tight_layout()
plt.show()


## Analysis & Conclusions

### 1. Each backend targets a specific mathematical structure

| Backend | Targets | Complexity | Approximation? |
|---|---|---|---|
| `direct` | any (fallback) | O(N²) / O(N⁴) | exact |
| `lowrank` | smooth joint kernels | O(r·N log N) | yes (Chebyshev/SVD) |
| `nufft` | oscillatory phases `c(x)g(ξ)e^{iλ(x)μ(ξ)}` | O(N log N) | exact (periodic only) |
| `aaa` | rational / pole-shaped symbols | O(N log N) | yes (rational fit) |
| `auto` | dispatches to the best of the above | — | — |
| `hybrid` | per-term routing for mixed symbols | O(N log N) per term | — |

### 2. Running the wrong backend wastes time

When a backend cannot represent the symbol's structure, it falls back to
direct quadrature. This fallback is **correct** but **expensive**: the time
spent attempting the failed decomposition is wasted. For example:

- NUFFT on `1/(1+x²+ξ²)`: no oscillatory phase found → fallback to direct.
- AAA on `sin(xξ)`: rational fit fails on rapid oscillations → fallback to direct.
- Lowrank on `sin(xξ)`: many Chebyshev terms needed → poor compression.

The targeted approach avoids all of these wasted attempts.

### 3. Auto-selection works correctly for single-class symbols

`joint_backend='auto'` inspects the symbolic structure and dispatches to
the best backend. For single-class symbols (purely oscillatory, purely
rational, purely smooth), auto correctly identifies the target and achieves
the same performance as the explicit backend.

### 4. Hybrid routing is essential for mixed symbols

For symbols containing terms of **different** structural classes,
monolithic `auto` may fail to find a global pattern and fall back to direct.
`apply_hybrid()` solves this by splitting the joint residual into additive
terms and routing each independently — recovering O(N log N) across the board.

### 5. Practical guidance

- **Local/separable only**: `apply_peetre` reproduces `direct` to machine
  precision at O(N log N) cost — no trade-off.
- **Oscillatory joint**: use `nufft` (exact, periodic only).
- **Rational/pole joint**: use `aaa` (check `joint_max_rel_error`).
- **Smooth joint**: use `lowrank` (tune `joint_degree`/`joint_tol`).
- **Mixed joint**: use `apply_hybrid()` for per-term optimal routing.
- **Unknown structure**: use `auto` and inspect
  `peetre_decomposition(classify_joint=True)` to verify the selection.


## Peetre-Only Scaling: Time vs. N (1D & 2D)

The benchmarks above always include `backend="direct"` as the ground truth,
which caps how large `N` can realistically go. Here we drop `direct` from
the loop entirely and time **only** `backend="peetre"` with the **target**
joint backend for each symbol, over a wider range of `N`.


### 1D Benchmarking


In [ ]:
def get_target_joint_backend(name):
    """Return the target joint backend for a given symbol name."""
    if 'Smooth' in name:
        return 'lowrank'
    elif 'Resolvent' in name or 'Mixed' in name:
        return 'aaa'
    elif 'Oscillatory' in name or 'Chirp' in name:
        return 'nufft'
    elif 'AllThree' in name:
        return 'auto'  # will fall back; hybrid is tested separately
    else:
        return 'direct'  # no joint residual


def benchmark_peetre_only(op, u, x_grid, kx, y_grid=None, ky=None,
                          joint_backend='direct',
                          repeats=3, boundary_condition='periodic'):
    """
    Time backend='peetre' only, for a single joint_backend.
    No error is computed (no 'direct' ground truth is run).

    Returns
    -------
    dict: joint_backend -> {'mean_time': float, 'times': list[float]}
    """
    kwargs = dict(backend='peetre', joint_backend=joint_backend)

    # Warm-up (symbolic decomposition caching, etc.)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        op.apply(u, x_grid, kx, y_grid=y_grid, ky=ky,
                 boundary_condition=boundary_condition, **kwargs)

    times = []
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for _ in range(repeats):
            t0 = time.perf_counter()
            op.apply(u, x_grid, kx, y_grid=y_grid, ky=ky,
                     boundary_condition=boundary_condition, **kwargs)
            times.append(time.perf_counter() - t0)

    return {joint_backend: {'mean_time': float(np.mean(times)), 'times': times}}


In [ ]:
N_values_1d_peetre = [16, 32, 64, 128, 256, 512]

rows_1d_peetre = []

print('Running 1D peetre-only scaling (targeted backends)...')
for N in N_values_1d_peetre:
    L = 10
    dx = L / N
    x_grid = -L/2 + dx * np.arange(N)
    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)

    for name, op in ops_1d.items():
        u = make_test_function(op, x_grid)
        jb = get_target_joint_backend(name)
        res = benchmark_peetre_only(op, u, x_grid, kx, repeats=1,
                                    joint_backend=jb,
                                    boundary_condition=boundary_condition)

        for jb_key, r in res.items():
            rows_1d_peetre.append(dict(N=N, symbol=name, joint_backend=jb_key,
                                       time=r['mean_time']))
        print(f"N={N:5d} | {name:24s} | joint={jb:8s} | "
              f"time: {list(res.values())[0]['mean_time']:.5f}s")

df_1d_peetre = pd.DataFrame(rows_1d_peetre)


In [ ]:
pivot_time_1d_peetre = df_1d_peetre.pivot_table(index=['symbol', 'N'],
                                                 columns='joint_backend', values='time')
display(pivot_time_1d_peetre)

fig, ax = plt.subplots(figsize=(10, 7))
names_1d = list(ops_1d.keys())
jb_style = {
    'direct':  ('-',  'o'),
    'lowrank': (':',  '^'),
    'nufft':   ('--', 's'),
    'aaa':     ('-.', 'D'),
    'auto':    ((0, (3, 1, 1, 1)), 'v'),
}

for name in names_1d:
    sub = df_1d_peetre[df_1d_peetre['symbol'] == name]
    for jb, (ls, mk) in jb_style.items():
        s = sub[sub['joint_backend'] == jb].sort_values('N')
        if len(s) == 0:
            continue
        ax.loglog(s['N'], s['time'], marker=mk, linestyle=ls,
                  color=colors_1d[name], label=f'{name} (joint={jb})', alpha=0.85)

# Reference O(N log N) slope
N_ref = np.array(N_values_1d_peetre)
first = df_1d_peetre[(df_1d_peetre['symbol'] == names_1d[0]) &
                     (df_1d_peetre['joint_backend'] == 'direct')].sort_values('N')
if len(first) > 0:
    scale = first['time'].iloc[-1] / (N_ref[-1] * np.log(N_ref[-1]))
    ax.loglog(N_ref, scale * N_ref * np.log(N_ref), 'k--', alpha=0.4,
              label=r'$O(N\log N)$ ref.')

ax.set_xlabel('Grid Size N')
ax.set_ylabel('Execution Time (s)')
ax.set_title('1D backend="peetre" only: Time vs. N (targeted joint backends)')
ax.legend(fontsize=5, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()


### 2D Benchmarking


In [ ]:
N_values_2d_peetre = [16, 32, 64, 128, 256]

rows_2d_peetre = []

print('Running 2D peetre-only scaling (targeted backends)...')
for N in N_values_2d_peetre:
    L = 10
    dx = L / N
    x_grid = -L/2 + dx * np.arange(N)
    y_grid = -L/2 + dx * np.arange(N)

    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(N, d=dx)

    for name, op in ops_2d.items():
        u = make_test_function(op, x_grid, y_grid)
        jb = get_target_joint_backend(name)
        res = benchmark_peetre_only(op, u, x_grid, kx, y_grid=y_grid, ky=ky,
                                    repeats=2, joint_backend=jb,
                                    boundary_condition=boundary_condition)

        for jb_key, r in res.items():
            rows_2d_peetre.append(dict(N=N, symbol=name, joint_backend=jb_key,
                                       time=r['mean_time']))
        print(f"N={N:4d}x{N:<4d} | {name:24s} | joint={jb:8s} | "
              f"time: {list(res.values())[0]['mean_time']:.5f}s")

df_2d_peetre = pd.DataFrame(rows_2d_peetre)


In [ ]:
pivot_time_2d_peetre = df_2d_peetre.pivot_table(index=['symbol', 'N'],
                                                  columns='joint_backend', values='time')
display(pivot_time_2d_peetre)

fig, ax = plt.subplots(figsize=(10, 7))
names_2d = list(ops_2d.keys())

for name in names_2d:
    sub = df_2d_peetre[df_2d_peetre['symbol'] == name]
    for jb, (ls, mk) in jb_style.items():
        s = sub[sub['joint_backend'] == jb].sort_values('N')
        if len(s) == 0:
            continue
        ax.loglog(s['N'], s['time'], marker=mk, linestyle=ls,
                  color=colors_2d[name], label=f'{name} (joint={jb})', alpha=0.85)

# Reference O(N² log N) slope (total points = N²)
N_ref2 = np.array(N_values_2d_peetre)
first2 = df_2d_peetre[(df_2d_peetre['symbol'] == names_2d[0]) &
                       (df_2d_peetre['joint_backend'] == 'direct')].sort_values('N')
if len(first2) > 0:
    scale2 = first2['time'].iloc[-1] / (N_ref2[-1]**2 * np.log(N_ref2[-1]))
    ax.loglog(N_ref2, scale2 * N_ref2**2 * np.log(N_ref2), 'k--', alpha=0.4,
              label=r'$O(N^2\log N)$ ref.')

ax.set_xlabel('Grid Size N (Total Points = N²)')
ax.set_ylabel('Execution Time (s)')
ax.set_title('2D backend="peetre" only: Time vs. N (targeted joint backends)')
ax.legend(fontsize=5, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()
